# Подготовка и очистка данных

**Проект:** Анализ международной образовательной мобильности (2000–2024).

**Источник данных:** Институт статистики ЮНЕСКО (UNESCO Institute for Statistics)

## Описание исходных данных

Исходный набор данных содержит:
- информацию о принимающих странах;
- временной ряд (2000–2024 гг.);
- численность российских студентов по странам за каждый год.

## Загрузка и первичный обзор данных

Загружен исходный CSV-файл, проведен первичный обзор структуры данных
(названия столбцов, типы данных, наличие пропусков и аномальных значений, релевантность данных).  
Цель — понять исходную структуру данных и определить необходимые шаги по их очистке.


In [1]:
import pandas as pd
df=pd.read_csv('../data/raw/student_mobility.csv')

df.head()

,indicatorId,geoUnit,year,value,qualifier,magnitude
0,26604,ABW,2003,0.0,NaN,NIL
1,26604,ABW,2006,0.0,NaN,NIL
2,26604,ABW,2009,0.0,NaN,NIL
3,26604,ABW,2010,0.0,NaN,NIL
4,26604,ABW,2011,0.0,NaN,NIL


## Очистка и проверка качества данных

Шаги:
- удаление лишних столбцов;
- переименование столбцов;
- удаление информации о странах, не принимавших российских студентов;
- приведение числовых значений в одинаковый формат;
- замена кодов стран ISO на полные названия стран.

In [2]:
df=df[['geoUnit', 'year', 'value']].copy()
df.columns=['Страна (ISO)','Год','Число студентов из РФ']

countries_with_students=df.groupby('Страна (ISO)')['Число студентов из РФ'].sum().loc[lambda x: x > 0].index
df=df[df['Страна (ISO)'].isin(countries_with_students)]

df['Число студентов из РФ']=df['Число студентов из РФ'].round().astype('Int64')

df

,Страна (ISO),Год,Число студентов из РФ
0,ABW,2003,0
1,ABW,2006,0
2,ABW,2009,0
3,ABW,2010,0
4,ABW,2011,0
...,...,...,...
2068,ZAF,2019,27
2069,ZAF,2020,20
2070,ZAF,2021,19
2071,ZAF,2022,13


In [3]:
import pycountry

def country_name(code):
    country=pycountry.countries.get(alpha_3=code)
    return country.name if country else code

df['Страна']=df['Страна (ISO)'].apply(country_name)

df=df.drop(columns=['Страна (ISO)'])
df=df[['Страна', 'Год', 'Число студентов из РФ']]

df

,Страна,Год,Число студентов из РФ
0,Aruba,2003,0
1,Aruba,2006,0
2,Aruba,2009,0
3,Aruba,2010,0
4,Aruba,2011,0
...,...,...,...
2068,South Africa,2019,27
2069,South Africa,2020,20
2070,South Africa,2021,19
2071,South Africa,2022,13


В ходе подготовки данных пропущенные значения не заполнялись. Отсутствие данных по количеству студентов из РФ за отдельные годы по некоторым странам интерпретируется как отсутствие информации, а не как нулевые значения.

## Проверка дубликатов

Проверено наличие дублирующихся записей «страна — год».

In [4]:
duplicates=df.duplicated(subset=['Страна', 'Год']).sum()
duplicates

np.int64(0)

## Проверка временного диапазона

Проверено, соответствует ли временной диапазон данных периоду с 2000 по 2024 гг. Данные за 2025 год исключены из анализа, поскольку информация за этот год есть только по одной стране (Грузии), что не позволяет проводить сравнительный анализ.

In [5]:
print(df['Год'].min(), df['Год'].max())
df_2025=df[df['Год']==2025]
df_2025

2000 2025


,Страна,Год,Число студентов из РФ
749,Georgia,2025,653


In [6]:
df=df[df['Год']<=2024]
df['Год'].min(), df['Год'].max()

(np.int64(2000), np.int64(2024))

## Проверка корректности значений

Проверено, что в столбце "Число студентов из РФ" нет отрицательных значений.

In [7]:
neg=df[df['Число студентов из РФ'] < 0]
neg

,Страна,Год,Число студентов из РФ


In [8]:
df.sample(10)

,Страна,Год,Число студентов из РФ
2000,United States,2018,5074
1212,Luxembourg,2016,49
1447,Malaysia,2014,168
1792,Slovakia,2012,25
899,India,2022,23
113,Austria,2014,939
2017,Uzbekistan,2019,220
1023,Japan,2019,521
857,Hungary,2009,150
1934,Tunisia,2001,0


## Итоговая информация

После очистки и подготовки данных сформирован итоговый датасет для дальнейшего анализа.

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1805 entries, 0 to 2072
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Страна                 1805 non-null   object
 1   Год                    1805 non-null   int64 
 2   Число студентов из РФ  1805 non-null   Int64 
dtypes: Int64(1), int64(1), object(1)
memory usage: 58.2+ KB


## Сохранение очищенных данных

Очищенный датасет сохранен в отдельный файл.

In [10]:
df.to_csv('../data/processed/student_mobility_clean.csv', index=False)

## Дальнейшие шаги анализа

- Разведочный анализ данных (EDA)
- Сравнительный анализ ведущих направлений мобильности - Европы и Азии.
- Факторный анализ.
- Прогноз трендов мобильности по странам до 2030 г.